# Week 4 Exercise – Python to C++ Code Generator (iyanuashiri)

Convert Python code to C++ via **OpenRouter** and **Gradio**: side-by-side editor, **model switch**, **streaming**, **compile & run**, **download .cpp**, **analytics** (lines in/out, time, model).

## Imports and environment

In [ ]:
import os
import re
import time
import tempfile
import subprocess
import shutil
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
try:
    from decouple import config
    OPEN_ROUTER_API_KEY = config("OPEN_ROUTER_API_KEY")
except Exception:
    OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")

OPEN_ROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=OPEN_ROUTER_BASE_URL, api_key=OPEN_ROUTER_API_KEY)

# OpenRouter models: latest from OpenAI, Anthropic, Google (check https://openrouter.ai/models for current list)
MODELS = [
    "google/gemini-3.1-pro-preview",
    "openai/gpt-5.4-pro",
    "openai/gpt-4o",
    "openai/gpt-4o-mini",
    "anthropic/claude-3.5-sonnet",
    "anthropic/claude-3.5-haiku",
    "anthropic/claude-3-opus",
    "google/gemini-2.0-flash-001",
]

## Prompts and conversion

In [ ]:
SYSTEM_PROMPT = """You convert Python code to high-performance C++17 code.
Output only the C++ code. No markdown, no code fences, no explanation.
Use modern C++, STL where appropriate, minimal dependencies. Produce identical output to the Python when run."""


def build_user_prompt(python_code: str, instructions: str = "") -> str:
    extra = f"\nAdditional instructions: {instructions}" if instructions.strip() else ""
    return f"""Convert this Python code to C++17. Output only the C++ source code, nothing else.{extra}

Python code:

```python
{python_code}
```"""


def strip_markdown(text: str) -> str:
    """Remove markdown code fences and leading/trailing whitespace."""
    if not text:
        return ""
    text = text.strip()
    for pattern in [r"^```cpp\s*", r"^```c\+\+\s*", r"^```\s*", r"\s*```\s*$"]:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return text.strip()


def messages_for(python_code: str, instructions: str = "") -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(python_code, instructions)},
    ]

In [ ]:
def convert_stream(python_code: str, instructions: str, model_id: str, temperature: float):
    """Stream conversion: yield (analytics_str, cpp_str). Outputs order is [analytics_out, cpp_out] so C++ streams into second box."""
    if not OPEN_ROUTER_API_KEY:
        yield "", "Set OPEN_ROUTER_API_KEY in .env"
        return
    if not (python_code or "").strip():
        yield "", "Enter Python code to convert."
        return
    lines_in = len(python_code.strip().splitlines())
    start = time.perf_counter()
    messages = messages_for(python_code, instructions)
    try:
        stream = openrouter.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=float(temperature),
            max_tokens=8192,
            stream=True,
        )
        accumulated = ""
        for chunk in stream:
            if not getattr(chunk, "choices", None) or len(chunk.choices) == 0:
                continue
            delta = chunk.choices[0].delta.content or ""
            if delta:
                accumulated += delta
                elapsed = time.perf_counter() - start
                analytics = f"Lines in: {lines_in} | Lines out: {len(accumulated.splitlines())} | Time: {elapsed:.2f}s | Model: {model_id}"
                # Yield raw accumulated during stream so the C++ box always shows content
                yield analytics, accumulated
        cpp_final = strip_markdown(accumulated)
        elapsed = time.perf_counter() - start
        lines_out = len(cpp_final.splitlines())
        analytics_final = f"Lines in: {lines_in} | Lines out: {lines_out} | Time: {elapsed:.2f}s | Model: {model_id}"
        yield analytics_final, cpp_final
    except Exception as e:
        yield f"API error: {e}", ""

## Compile, run, and download

In [ ]:
def detect_compiler() -> list | None:
    """Return [compiler, ...args] for g++ or clang++, or None if not found."""
    for name in ["g++", "clang++"]:
        path = shutil.which(name)
        if path:
            return [path, "-std=c++17", "-O2", "-Wall", "main.cpp", "-o", "main"]
    return None


def compile_cpp(cpp_code: str) -> str:
    """Compile C++ in a temp dir. Return status message."""
    if not (cpp_code or "").strip():
        return "No C++ code to compile."
    compiler_cmd = detect_compiler()
    if not compiler_cmd:
        return "No g++ or clang++ found. Install a C++ compiler or use an online compiler."
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir)
        (path / "main.cpp").write_text(cpp_code, encoding="utf-8")
        try:
            result = subprocess.run(
                compiler_cmd,
                cwd=tmpdir,
                capture_output=True,
                text=True,
                timeout=30,
            )
            if result.returncode != 0:
                return f"Compile failed:\n{result.stderr or result.stdout}"
            return "Compiled successfully."
        except subprocess.TimeoutExpired:
            return "Compilation timed out."
        except Exception as e:
            return f"Compile error: {e}"


def run_cpp(cpp_code: str) -> str:
    """Compile and run C++ in a temp dir; return stdout+stderr or error message."""
    if not (cpp_code or "").strip():
        return "No C++ code to run."
    compiler_cmd = detect_compiler()
    if not compiler_cmd:
        return "No g++ or clang++ found. Install a C++ compiler or use an online compiler."
    is_windows = os.name == "nt"
    exe_name = "main.exe" if is_windows else "main"
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir)
        (path / "main.cpp").write_text(cpp_code, encoding="utf-8")
        try:
            r = subprocess.run(compiler_cmd, cwd=tmpdir, capture_output=True, text=True, timeout=30)
            if r.returncode != 0:
                return f"Compile failed:\n{r.stderr or r.stdout}"
            exe_path = path / exe_name
            if not exe_path.exists():
                return "Compiler did not produce executable."
            run_result = subprocess.run(
                [str(exe_path)] if is_windows else ["./main"],
                cwd=tmpdir,
                capture_output=True,
                text=True,
                timeout=10,
            )
            out = (run_result.stdout or "").strip()
            err = (run_result.stderr or "").strip()
            return out + ("\n" + err if err else "") or "(no output)"
        except subprocess.TimeoutExpired as e:
            return "Compilation or run timed out."
        except Exception as e:
            return f"Error: {e}"


def download_cpp(cpp_code: str) -> str | None:
    """Write C++ to a temp file and return path for gr.File download."""
    if not (cpp_code or "").strip():
        return None
    tmp = Path(tempfile.gettempdir()) / "generated_main.cpp"
    tmp.write_text(cpp_code, encoding="utf-8")
    return str(tmp)

## Default example and Gradio UI

In [ ]:
DEFAULT_PYTHON = '''def greet(name):
    print(f"Hello, {name}!")

for i in range(3):
    greet("User" + str(i))
'''

In [ ]:
with gr.Blocks(title="Python to C++", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Python → C++ code generator\nPaste Python code; pick model and options; convert. Then compile, run, or download.")
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            python_in = gr.Textbox(
                label="Python (input)",
                value=DEFAULT_PYTHON,
                lines=16,
                placeholder="Paste Python code here...",
            )
            clear_py_btn = gr.Button("Clear Python", size="sm")
        with gr.Column(scale=1):
            cpp_out = gr.Textbox(
                label="C++ (generated) — select and copy if needed",
                lines=16,
                placeholder="C++ will appear here (streaming)...",
            )
            clear_cpp_btn = gr.Button("Clear C++", size="sm")

    with gr.Row():
        instructions_in = gr.Textbox(
            label="Optional instructions (e.g. 'Use C++17', 'add comments')",
            placeholder="Optional...",
            lines=1,
        )
    with gr.Row():
        model_in = gr.Dropdown(choices=MODELS, value=MODELS[0], label="Model")
        temperature_in = gr.Slider(0.1, 0.9, value=0.3, step=0.1, label="Temperature")
        convert_btn = gr.Button("Convert", variant="primary")

    analytics_out = gr.Textbox(label="Analytics", interactive=False, lines=4)
    with gr.Row():
        compile_btn = gr.Button("Compile")
        run_btn = gr.Button("Run C++")
        download_btn = gr.Button("Download .cpp")
    compile_status = gr.Textbox(label="Compile result", interactive=False, lines=4)
    run_status = gr.Textbox(label="Run result", interactive=False, lines=6)
    download_file = gr.File(label="Download")

    convert_btn.click(
        fn=convert_stream,
        inputs=[python_in, instructions_in, model_in, temperature_in],
        outputs=[analytics_out, cpp_out],
        queue=True,
    )
    clear_py_btn.click(fn=lambda: "", outputs=python_in)
    clear_cpp_btn.click(fn=lambda: "", outputs=cpp_out)
    compile_btn.click(fn=compile_cpp, inputs=cpp_out, outputs=compile_status)
    run_btn.click(fn=run_cpp, inputs=cpp_out, outputs=run_status)
    download_btn.click(fn=download_cpp, inputs=cpp_out, outputs=download_file)

demo.launch(share=True)